# Ćwiczenia 3 — Uczenie nienadzorowane: PCA, mapy próbek, skupienia (wersja Python)
**Dane demo: kohorta T2D (QinJ_2012)**

Na Ćwiczeniach 2 oglądaliśmy cechy pojedynczo. Teraz patrzymy na wszystkie cechy jednocześnie i pytamy, czy próbki układają się w jakąś strukturę: czy tworzą grupy, jakie kierunki zmienności dominują w danych i czy ta struktura ma związek z klasą. Metody nienadzorowane (ang. *unsupervised learning*) nie korzystają z etykiet: klasę nanosimy na wyniki dopiero na końcu, żeby sprawdzić, czy największa zmienność w danych wiąże się z chorobą. Jeśli tak, model nadzorowany ma łatwe zadanie; jeśli nie, różnice między klasami są subtelne i model musi ich szukać wśród wielu innych źródeł zmienności.

Pracujemy na całym zbiorze, bez podziału na trening i test. Przy oglądaniu danych bez etykiet wolno tak robić, bo niczego nie dopasowujemy do klasy i niczego nie oceniamy. Decyzje dotyczące modelu (np. wybór przekształcenia albo liczby składowych jako cech) podejmuje się później, wyłącznie na zbiorze treningowym.

## Wczytanie danych

Wczytujemy te same pliki co na Ćwiczeniach 2 i zostawiamy cechy obecne w co najmniej 10% próbek. Notatnik trzeba zapisać w tym samym katalogu co notatniki z Ćwiczeń 1 i 2, czyli obok katalogu `dane/`. Notatnik najlepiej uruchamiać od początku po ponownym uruchomieniu jądra (*Kernel → Restart*), komórka po komórce.

Biblioteki metod nienadzorowanych instaluje się jednorazowo w terminalu (jeśli `import` zgłasza błąd `No module named ...`):

```
pip install scipy scikit-learn umap-learn
```

In [ ]:
import warnings                                      # obsługa ostrzeżeń
warnings.filterwarnings("ignore")                    # ukrycie ostrzeżeń informacyjnych bibliotek (np. umap)
import numpy as np                                   # obliczenia na tablicach
import pandas as pd                                  # tabele danych
import matplotlib.pyplot as plt                      # wykresy
from scipy.spatial.distance import pdist, squareform # odległości między próbkami
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster   # grupowanie hierarchiczne
from sklearn.decomposition import PCA                # analiza głównych składowych
from sklearn.preprocessing import StandardScaler     # standaryzacja (do 🔧)
from sklearn.manifold import TSNE                    # t-SNE
from sklearn.cluster import KMeans                   # k-średnich
from sklearn.metrics import silhouette_score, adjusted_rand_score   # sylwetka i ARI
import umap                                          # UMAP (pakiet umap-learn)
pd.set_option("display.width", 120)                  # szersze wydruki tabel

# index_col=0: pierwsza kolumna pliku to nazwy próbek
X = pd.read_csv("dane/t2d_features.csv", index_col=0)
y = pd.read_csv("dane/t2d_labels.csv")["grupa"].to_numpy()   # tablica napisów "control"/"T2D"

Xf = X.loc[:, (X > 0).mean() >= 0.10]                # cechy obecne w co najmniej 10% próbek
print(Xf.shape)                                      # (liczba próbek, liczba cech)
print(pd.Series(y).value_counts())                   # liczba próbek w każdej klasie

cols = {"control": "#2C5F7C", "T2D": "#D55E00"}      # kolory klas na wszystkich rycinach (granatowy, pomarańczowy)

## 1. Przekształcenie CLR

Na Ćwiczeniach 2 okazało się, że na surowych danych odległości między próbkami wyznacza w dużej mierze jedna cecha (*Prevotella copri*), bo ma największy rozrzut. Metody z tego arkusza opierają się na odległościach i wariancji, więc bez przekształcenia pokazywałyby głównie tę jedną cechę. Do tego dane są kompozycyjne (ang. *compositional data*): wartości w każdej próbce sumują się do 100%, więc wzrost jednej cechy oznacza spadek pozostałych. Przekształcenie CLR (ang. *centered log-ratio*) rozwiązuje oba problemy: każdą wartość logarytmuje się i odejmuje od niej średnią logarytmów w tej samej próbce. Wynik mówi, ile razy cecha jest liczniejsza od „typowej” cechy w danej próbce, więc liczą się proporcje, a nie wartości bezwzględne. Logarytm z zera nie istnieje, dlatego przed logarytmowaniem dodajemy małą stałą `1e-6`, czyli pseudoliczność (ang. *pseudocount*), tę samą co w całym kursie.

In [ ]:
# własna funkcja: CLR dla każdego wiersza (próbki) tabeli M
def clr(M):
    L = np.log(M.to_numpy() + 1e-6)                  # logarytm naturalny po dodaniu pseudoliczności
    return L - L.mean(axis=1, keepdims=True)         # od każdej wartości odejmujemy średnią logarytmów jej wiersza

Z = clr(Xf)                                          # dane po CLR (tablica NumPy): te same próbki i cechy
print(Z.shape)                                       # wymiary się nie zmieniają

variances = np.sort(Z.var(axis=0, ddof=1))[::-1]     # wariancja każdej cechy po CLR, od największej
print(round(variances[:5].sum() / variances.sum(), 3))   # udział 5 cech o największej wariancji
d_top1 = pdist(Xf[["species:Prevotella copri"]])     # odległości liczone tylko na P. copri
print(np.corrcoef(pdist(Z), d_top1)[0, 1])           # na ile ta cecha wyznacza odległości po CLR

**Na co patrzeć.** Na surowych danych pięć cech skupiało około 60% całkowitej wariancji, a po CLR skupia tylko kilka procent. Korelacja odległości z odległością liczoną na samej *P. copri* spadła z około 0,70 (Ćwiczenia 2) do wartości bliskiej zeru: żadna pojedyncza cecha nie wyznacza już podobieństwa próbek.

## 2. Analiza głównych składowych

Analiza głównych składowych (ang. *principal component analysis*, PCA) szuka kierunków, wzdłuż których próbki różnią się najbardziej. Pierwsza składowa główna (PC1) to kierunek największej zmienności, druga (PC2) kolejny, prostopadły do pierwszego, i tak dalej. Po co: dwie pierwsze składowe pozwalają pokazać próbki opisane 190 cechami na jednej płaskiej rycinie, a odsetek wariancji mówi, jaką część informacji ta rycina zachowuje. PCA liczymy na danych CLR po wycentrowaniu, bez standaryzacji, jak na wykładzie 3: po CLR wszystkie cechy mają wspólną jednostkę (logarytm ilorazu).

In [ ]:
pca = PCA()                                          # PCA() sama centruje dane; nie standaryzuje
scores = pca.fit_transform(Z)                        # położenie próbek na składowych (🔧: StandardScaler().fit_transform(Z))
var_prop = pca.explained_variance_ratio_             # odsetek wariancji każdej składowej
print(var_prop[:5].round(3))                         # pięć pierwszych składowych
print(np.cumsum(var_prop)[[1, 9]].round(3))          # odsetek skumulowany dla 2 i 10 składowych (numeracja od 0)

Wykres osypiska (ang. *scree plot*) pokazuje odsetek wariancji kolejnych składowych:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))   # nowa rycina; figsize: szerokość i wysokość w calach
ax.bar(range(1, 21), 100 * var_prop[:20], color="grey")   # słupek dla każdej z 20 pierwszych składowych
ax.set_xticks([1, 5, 10, 15, 20])                   # podziałka osi w liczbach całkowitych
ax.set_xlabel("Składowa główna"); ax.set_ylabel("Wariancja wyjaśniona (%)")   # podpisy osi
plt.show()   # wyświetlenie ryciny

**Na co patrzeć.** PC1 i PC2 wyraźnie odstają od pozostałych, a dalej słupki maleją powoli. Wariancja rozkłada się na wiele kierunków, więc rycina z dwiema osiami pokazuje tylko niewielką część informacji zawartej w danych.

Wykres próbek na dwóch pierwszych składowych. PCA nie zna etykiet; kolor klasy nanosimy po obliczeniach:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))   # nowa rycina; figsize: szerokość i wysokość w calach
for group, colour in cols.items():                   # osobno każda klasa, żeby mieć legendę
    k = y == group                                   # True dla próbek tej klasy
    ax.scatter(scores[k, 0], scores[k, 1], facecolors="none", edgecolors=colour, label=group)   # puste kółka w kolorze klasy
ax.set_xlabel(f"PC1 ({100 * var_prop[0]:.1f}%)")     # odsetek wariancji w podpisie osi
ax.set_ylabel(f"PC2 ({100 * var_prop[1]:.1f}%)")   # f"...": napis z wstawioną liczbą
ax.legend()   # legenda z nazwami klas
plt.show()   # wyświetlenie ryciny

Ładunki (ang. *loadings*) mówią, z jaką wagą każda cecha wchodzi do składowej. Pięć cech o największym ładunku PC1 (bez względu na znak, bo znak składowej jest umowny i może się różnić między programami):

In [ ]:
load_pc1 = pd.Series(pca.components_[0], index=Xf.columns)   # ładunki PC1: jedna liczba na cechę
top5 = load_pc1.abs().sort_values(ascending=False).index[:5] # pięć największych wartości bezwzględnych
print(load_pc1[top5].round(3).rename(lambda n: n.replace("species:", "")))   # ładunki z krótkimi nazwami cech

**Na co patrzeć.** Cechy o przeciwnych znakach działają w przeciwnych kierunkach: próbki z jednego końca PC1 mają względnie więcej cech o ładunku dodatnim, a z drugiego końca więcej cech o ładunku ujemnym. Ładunek opisuje udział cechy w zmienności danych, a nie jej związek z chorobą.

🔧 **Do sprawdzenia.** W komórce z PCA zamienić `pca.fit_transform(Z)` na `pca.fit_transform(StandardScaler().fit_transform(Z))` (standaryzacja: każda cecha z tą samą wagą) i uruchomić sekcję ponownie. Jak zmienia się odsetek wariancji PC1 i wygląd wykresu próbek?

> **✏️ Zadanie 1.** Odczytać, jaki odsetek wariancji wyjaśniają razem PC1 i PC2 i ile wyjaśnia 10 pierwszych składowych. Czy na wykresie PC1–PC2 klasy tworzą oddzielne chmury, czy się nakładają? Wzdłuż której osi różnią się bardziej? Odpowiedzieć w 2–3 zdaniach.

> **✏️ Zadanie 2.** Sprawdzić, co opisuje PC1: policzyć korelację położenia próbek na PC1 z liczbą cech niezerowych w próbce, `np.corrcoef(scores[:, 0], (Xf > 0).sum(axis=1))[0, 1]`. Co oznacza ta wartość? Wskazówka do interpretacji: po CLR z pseudolicznością 1e-6 każde zero staje się bardzo małą liczbą (logarytm z 1e-6 to około −14).

In [ ]:
# miejsce na obliczenie z Zadania 2

## 3. Analiza współrzędnych głównych na odległości Braya–Curtisa

PCA opiera się na odległości euklidesowej między próbkami. W danych o składzie próbek często stosuje się inną miarę: odległość Braya–Curtisa (ang. *Bray–Curtis dissimilarity*), liczoną na udziałach procentowych; 0 oznacza identyczny skład, 1 brak wspólnych cech. Analiza współrzędnych głównych (ang. *principal coordinates analysis*, PCoA) rysuje próbki tak, by odległości na rycinie jak najlepiej odtwarzały wybraną miarę. Po co: inna miara odległości może pokazać inną strukturę danych, więc warto sprawdzić, czy wnioski z PCA się powtarzają. W Pythonie PCoA liczymy kilkoma wierszami NumPy (to samo robi funkcja `cmdscale()` w R).

In [ ]:
D = squareform(pdist(Xf, metric="braycurtis"))       # macierz odległości Braya–Curtisa (na udziałach), próbki × próbki
n = D.shape[0]                                       # liczba próbek
J = np.eye(n) - np.ones((n, n)) / n                  # macierz centrująca
B = -0.5 * J @ (D ** 2) @ J                          # podwójnie wycentrowana macierz kwadratów odległości
eig, vec = np.linalg.eigh(B)                         # wartości i wektory własne (rosnąco)
order = np.argsort(eig)[::-1]                        # kolejność od największej wartości własnej
eig, vec = eig[order], vec[:, order]   # wartości i wektory własne w nowej kolejności
axis_prop = eig / eig[eig > 0].sum()                 # odsetek osi (względem sumy dodatnich wartości własnych)
print(axis_prop[:2].round(3))                        # odsetek dwóch pierwszych osi
points = vec[:, :2] * np.sqrt(eig[:2])               # położenie próbek na dwóch pierwszych osiach

fig, ax = plt.subplots(figsize=(6, 4.5))   # nowa rycina; figsize: szerokość i wysokość w calach
for group, colour in cols.items():   # osobno każda klasa, żeby mieć legendę
    k = y == group   # True dla próbek tej klasy
    ax.scatter(points[k, 0], points[k, 1], facecolors="none", edgecolors=colour, label=group)   # puste kółka w kolorze klasy
ax.set_xlabel(f"Oś 1 ({100 * axis_prop[0]:.1f}%)"); ax.set_ylabel(f"Oś 2 ({100 * axis_prop[1]:.1f}%)")   # podpisy osi z odsetkiem osi
ax.legend()   # legenda z nazwami klas
plt.show()   # wyświetlenie ryciny

# czy pierwsza oś PCoA wiąże się z udziałem najliczniejszej cechy (korelacja rang)
print(pd.Series(points[:, 0]).corr(pd.Series(Xf["species:Prevotella copri"].to_numpy()), method="spearman"))

**Na co patrzeć.** Odległość Braya–Curtisa liczona na udziałach procentowych zależy przede wszystkim od cech liczebnych, dlatego pierwsza oś PCoA wyraźnie wiąże się z udziałem *P. copri* (znak osi jest umowny). To inny obraz niż w PCA na danych CLR: wybór miary odległości zmienia to, co widać na mapie próbek.

## 4. Mapy t-SNE i UMAP

t-SNE (ang. *t-distributed stochastic neighbor embedding*) i UMAP (ang. *uniform manifold approximation and projection*) układają próbki na płaszczyźnie tak, żeby próbki bliskie w danych leżały blisko na mapie. Po co: lepiej niż PCA pokazują lokalne grupy podobnych próbek. Interpretuje się tylko sąsiedztwo; odległości między odległymi grupami i wielkości grup na mapie nie odpowiadają danym. Obie metody są losowe, więc ustalamy ziarno. Wejściem jest 10 pierwszych składowych PCA, co pomija składowe o małej wariancji i skraca obliczenia. Parametr `perplexity` w t-SNE to w przybliżeniu liczba sąsiadów, których bierze pod uwagę każda próbka; `n_neighbors` w UMAP pełni podobną rolę.

In [ ]:
scores10 = scores[:, :10]                            # 10 pierwszych składowych głównych
tsne_xy = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(scores10)   # random_state: ziarno
umap_xy = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(scores10)   # mapa UMAP; random_state: ziarno

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))    # dwa panele obok siebie
for ax, xy, title in [(axes[0], tsne_xy, "t-SNE (perplexity = 30)"), (axes[1], umap_xy, "UMAP (n_neighbors = 15)")]:   # jeden panel na metodę
    for group, colour in cols.items():   # osobno każda klasa, żeby mieć legendę
        k = y == group   # True dla próbek tej klasy
        ax.scatter(xy[k, 0], xy[k, 1], facecolors="none", edgecolors=colour, label=group)   # puste kółka w kolorze klasy
    ax.set_title(title); ax.set_xlabel("Wymiar 1"); ax.set_ylabel("Wymiar 2")   # tytuł panelu i podpisy osi
axes[1].legend()   # legenda w prawym panelu
plt.show()   # wyświetlenie ryciny

**Na co patrzeć.** Kolory pokazują, czy najbliżsi sąsiedzi próbki należą zwykle do tej samej klasy. Jeśli w małych fragmentach mapy kolory są wymieszane, sąsiedztwo w danych słabo wiąże się z chorobą. Skupiska widoczne na mapie należy traktować jako hipotezę do sprawdzenia, bo t-SNE i UMAP tworzą wyraźne skupiska także w danych bez wyraźnej struktury. Mapy w R i w Pythonie wyglądają inaczej, bo obie metody są losowe, a programy korzystają z różnych generatorów liczb losowych.

🔧 **Do sprawdzenia.** W wywołaniu `umap.UMAP()` zmienić `n_neighbors` na 5, a potem na 50.

> **✏️ Zadanie 3.** W wywołaniu `TSNE()` zmienić `perplexity` na 5, a potem na 50 (ziarno bez zmian) i za każdym razem narysować mapę ponownie. Jak zmienia się liczba i rozstaw skupisk? Czy przy którejś wartości klasy rozdzielają się wyraźniej? Co z tego wynika dla interpretacji jednej mapy t-SNE?

## 5. Analiza skupień

Analiza skupień (ang. *clustering*) dzieli próbki na grupy podobnych próbek. Po co: sprawdzamy, czy dane same dzielą się na grupy i czy te grupy odpowiadają klasom. Metoda k-średnich (ang. *k-means*) dzieli próbki na k skupień tak, by próbki leżały jak najbliżej środka swojego skupienia. Liczbę skupień wybieramy dwiema miarami: metodą łokcia (ang. *elbow method*; suma kwadratów odległości od środków skupień, WSS, maleje z każdym k i szuka się miejsca, po którym spada już wolno) i średnią szerokością sylwetki (ang. *silhouette width*; od −1 do 1, im wyżej, tym lepiej oddzielone skupienia). Skupienia wyznaczamy na tych samych 10 składowych PCA.

In [ ]:
ks = range(2, 7)                                     # sprawdzane liczby skupień: 2–6
wss, sil = [], []                                    # puste listy na wyniki
for k in ks:                                         # pętla po kolejnych wartościach k
    km_k = KMeans(n_clusters=k, n_init=25, random_state=42).fit(scores10)   # n_init=25: 25 startów, zostaje najlepszy
    wss.append(km_k.inertia_)                        # inertia_ to WSS dla tego k
    sil.append(silhouette_score(scores10, km_k.labels_))   # średnia sylwetka wszystkich próbek
print(pd.DataFrame({"k": list(ks), "WSS": np.round(wss), "silhouette": np.round(sil, 3)}))   # tabela wyników dla każdego k

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))   # dwa panele obok siebie
axes[0].plot(list(ks), wss, marker="o"); axes[0].set_title("WSS (metoda łokcia)")   # linia z punktami: WSS dla każdego k
axes[1].plot(list(ks), sil, marker="o"); axes[1].set_title("Średnia sylwetka")   # linia z punktami: sylwetka dla każdego k
for ax in axes: ax.set_xlabel("Liczba skupień k")   # podpis osi X w obu panelach
plt.show()   # wyświetlenie ryciny

**Na co patrzeć.** Krzywa WSS opada bez wyraźnego załamania, a średnia sylwetka jest największa dla k = 2, ale niska. Próbki nie tworzą wyraźnie oddzielonych grup; dalej porównujemy podział na dwa skupienia z dwiema klasami.

Grupowanie hierarchiczne (ang. *hierarchical clustering*) łączy kolejno najbliższe próbki i skupienia w drzewo, czyli dendrogram. Metoda Warda przy każdym łączeniu wybiera parę skupień, której połączenie najmniej zwiększa WSS. Przecięcie drzewa daje podział na skupienia.

In [ ]:
hc = linkage(scores10, method="ward")                # drzewo metodą Warda na odległościach euklidesowych
fig, ax = plt.subplots(figsize=(10, 4))   # nowa rycina; figsize: szerokość i wysokość w calach
dendrogram(hc, no_labels=True, color_threshold=0, above_threshold_color="black", ax=ax)   # bez nazw próbek
ax.set_ylabel("Wysokość"); ax.set_title("Dendrogram (metoda Warda)")   # podpis osi i tytuł
plt.show()   # wyświetlenie ryciny

Porównanie skupień z klasami. Skorygowany indeks Randa (ang. *adjusted Rand index*, ARI) mierzy zgodność dwóch podziałów tych samych próbek: 1 oznacza pełną zgodność, 0 zgodność taką jak przy podziale losowym.

In [ ]:
km = KMeans(n_clusters=2, n_init=25, random_state=42).fit(scores10)   # k-średnich dla k = 2
hc_cl = fcluster(hc, t=2, criterion="maxclust")      # przecięcie drzewa na 2 skupienia
print(pd.crosstab(km.labels_, y, rownames=["cluster"], colnames=["class"])[["control", "T2D"]])   # skupienie k-średnich × klasa; kolumny w kolejności control, T2D
print(pd.crosstab(hc_cl, y, rownames=["cluster"], colnames=["class"])[["control", "T2D"]])        # skupienie metody Warda × klasa
print(round(adjusted_rand_score(y, km.labels_), 3))  # ARI: k-średnich a klasa
print(round(adjusted_rand_score(y, hc_cl), 3))       # ARI: metoda Warda a klasa
print(round(adjusted_rand_score(km.labels_, hc_cl), 3))   # ARI: obie metody skupień między sobą

**Na co patrzeć.** Numery skupień są umowne i mogą się zamienić po zmianie ziarna albo między programami; ARI od numeracji nie zależy. Gdyby skupienia odpowiadały klasom, w każdym wierszu tabeli przeważałaby wyraźnie jedna klasa.

> **✏️ Zadanie 4.** Na podstawie tabel i wartości ARI: czy skupienia odpowiadają klasom? Czy obie metody skupień znajdują podobny podział? Policzyć medianę liczby cech niezerowych w każdym skupieniu k-średnich, `(Xf > 0).sum(axis=1).groupby(km.labels_).median()`, i ocenić, czym różnią się skupienia. Odpowiedzieć w 2–3 zdaniach.

In [ ]:
# miejsce na obliczenie z Zadania 4